In [3]:
# Run once — comment out after first install
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "spacy", "emoji", "scikit-learn", "pandas", "numpy", "joblib"
], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 22.1 MB/s eta 0:00:0000:010:01



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


CompletedProcess(args=['/Users/yatharthnehva/.pyenv/versions/3.11.8/bin/python', '-m', 'spacy', 'download', 'en_core_web_sm'], returncode=0)

In [4]:
import re
import glob
import random
import warnings
import numpy as np
import pandas as pd
import joblib
import spacy
import emoji

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')

# ── Adjust these to match your directory layout ──────────────────────────────
# This notebook lives in: characterlevelattacks/coreattacks/
ROOT = Path("../")   # characterlevelattacks/

TWITTER_CSV  = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/emojibased/twittersenti/twittersenti140.csv")
SPGC_DIR     = ROOT / "stylometric" / "SPGC"
BLOG_DIR     = ROOT / "stylometric" / "Blogdataset"
MODEL_SAVE   = Path("register_clf.joblib")   # saved alongside this notebook
# ─────────────────────────────────────────────────────────────────────────────

print("Twitter CSV exists:", TWITTER_CSV.exists())
print("SPGC dir exists:", SPGC_DIR.exists())
print("Blog dir exists:", BLOG_DIR.exists())

Twitter CSV exists: True
SPGC dir exists: True
Blog dir exists: True


In [5]:
class ZoneDetector:
    """
    Builds a character-level protection mask for an input string.
    Protected positions must not be perturbed by any downstream attack.
    """

    URL_RE = re.compile(r'https?://\S+|www\.\S+')

    def __init__(self, spacy_model: str = "en_core_web_sm"):
        self.nlp = spacy.load(spacy_model)

    # ------------------------------------------------------------------ #

    def detect(self, text: str) -> dict:
        """
        Parameters
        ----------
        text : str

        Returns
        -------
        dict with keys:
            char_mask       : np.ndarray[bool], shape=(len(text),)
                              True  → position is protected
                              False → position is eligible for perturbation
            token_eligible  : list[bool], one entry per spaCy token
            tokens          : list[str]
            protected_spans : list[tuple(start, end, reason)]
        """
        n = len(text)
        mask = np.zeros(n, dtype=bool)
        spans = []

        # ── 1. URLs ──────────────────────────────────────────────────── #
        for m in self.URL_RE.finditer(text):
            mask[m.start():m.end()] = True
            spans.append((m.start(), m.end(), 'url'))

        # ── 2. Existing emoji ────────────────────────────────────────── #
        for i, ch in enumerate(text):
            if emoji.is_emoji(ch):
                mask[i] = True
                spans.append((i, i + 1, 'emoji'))

        # ── 3. Named entities + proper nouns (spaCy) ─────────────────── #
        doc = self.nlp(text)

        for ent in doc.ents:
            mask[ent.start_char:ent.end_char] = True
            spans.append((ent.start_char, ent.end_char, f'ent:{ent.label_}'))

        for tok in doc:
            if tok.pos_ == 'PROPN':
                mask[tok.idx: tok.idx + len(tok.text)] = True
                spans.append((tok.idx, tok.idx + len(tok.text), 'propn'))

        # ── 4. Sentence-final punctuation ────────────────────────────── #
        for sent in doc.sents:
            last = sent[-1]
            if last.is_punct:
                mask[last.idx: last.idx + len(last.text)] = True
                spans.append((last.idx, last.idx + len(last.text), 'sent_punct'))

        # ── Build per-token eligibility ──────────────────────────────── #
        token_eligible = [
            not mask[tok.idx: tok.idx + len(tok.text)].any()
            for tok in doc
        ]

        return {
            'char_mask':       mask,
            'token_eligible':  token_eligible,
            'tokens':          [tok.text for tok in doc],
            'protected_spans': spans,
        }

    # ── Pretty-print helper ─────────────────────────────────────────── #

    def visualise(self, text: str) -> None:
        """Print text with protected characters marked in brackets."""
        result = self.detect(text)
        mask = result['char_mask']
        out, in_block = [], False
        for i, ch in enumerate(text):
            if mask[i] and not in_block:
                out.append('['); in_block = True
            elif not mask[i] and in_block:
                out.append(']'); in_block = False
            out.append(ch)
        if in_block:
            out.append(']')
        print(''.join(out))
        eligible = sum(result['token_eligible'])
        total    = len(result['token_eligible'])
        print(f"  → {eligible}/{total} tokens eligible for perturbation")
        print(f"  → Protected spans: {result['protected_spans']}\n")

In [6]:
# Testing the detector

zd = ZoneDetector()

tests = [
    "The quick brown fox jumps over the lazy dog.",
    "Check out https://github.com/Ynehra24/NLPproject for the code!",
    "Elon Musk announced Tesla's new model yesterday 😊.",
    "The Battle of Waterloo changed European history forever.",
    "I love this so much lol it's hilarious",
]

for t in tests:
    zd.visualise(t)

The quick brown fox jumps over the lazy dog[.]
  → 9/10 tokens eligible for perturbation
  → Protected spans: [(43, 44, 'sent_punct')]

Check out [https://github.com/Ynehra24/NLPproject] for the code[!]
  → 5/7 tokens eligible for perturbation
  → Protected spans: [(10, 48, 'url'), (61, 62, 'sent_punct')]

[Elon Musk] announced [Tesla]'s new model [yesterday] [😊.]
  → 4/10 tokens eligible for perturbation
  → Protected spans: [(48, 49, 'emoji'), (0, 9, 'ent:PERSON'), (20, 25, 'ent:ORG'), (38, 47, 'ent:DATE'), (0, 4, 'propn'), (5, 9, 'propn'), (20, 25, 'propn'), (49, 50, 'sent_punct')]

[The Battle of Waterloo] changed [European] history forever[.]
  → 3/9 tokens eligible for perturbation
  → Protected spans: [(0, 22, 'ent:WORK_OF_ART'), (31, 39, 'ent:NORP'), (4, 10, 'propn'), (14, 22, 'propn'), (55, 56, 'sent_punct')]

I love this so much lol it's hilarious
  → 9/9 tokens eligible for perturbation
  → Protected spans: []



In [7]:
# ── Data loaders ─────────────────────────────────────────────────────────── #

def load_twitter(path: Path, n: int = 15_000, seed: int = 42) -> list[str]:
    """
    Sentiment140 CSV format:
    target | id | date | flag | user | text
    Returns a sample of `n` tweet texts.
    """
    df = pd.read_csv(
        path, encoding='latin-1', header=None,
        names=['target', 'id', 'date', 'flag', 'user', 'text']
    )
    texts = df['text'].dropna().tolist()
    # Filter very short or empty strings
    texts = [t for t in texts if isinstance(t, str) and len(t.strip()) > 20]
    random.seed(seed)
    return random.sample(texts, min(n, len(texts)))


def load_spgc(path: Path, n: int = 15_000, seed: int = 42) -> list[str]:
    """
    Loads formal sentences from SPGC (Project Gutenberg) Parquet file.
    """
    df = pd.read_parquet(path)
    
    # Try to find the text column automatically
    col = next((c for c in df.columns
                if any(k in c.lower() for k in ('text', 'content', 'body'))),
               None)
    
    if col:
        texts = df[col].dropna().astype(str).tolist()
    else:
        # Fallback to the first column if typical names aren't found
        texts = df.iloc[:, 0].dropna().astype(str).tolist()

    texts = [t for t in texts if len(t.strip()) > 30]
    random.seed(seed)
    return random.sample(texts, min(n, len(texts)))


# Quick preview
print("Loading data...")
# Assuming TWITTER_CSV is defined earlier in your notebook
informal_texts = load_twitter(TWITTER_CSV) 

SPGC_PARQUET = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/stylometric/SPGC/gutenberg_en_20k.parquet")
formal_texts   = load_spgc(SPGC_PARQUET)

print(f"Informal samples : {len(informal_texts):,}")
print(f"Formal samples   : {len(formal_texts):,}")
print("\nInformal sample:", informal_texts[0])
print("Formal sample:  ", formal_texts[0])


Loading data...
Informal samples : 15,000
Formal samples   : 15,000

Informal sample: Wow! 25 years of #Tetris! Thank you Alexey for making this great puzzle game!  http://bit.ly/11icWS
Formal sample:   The Project Gutenberg eBook, Arthur Machen, by Vincent Starrett


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org





Title: Arthur Machen
       A Novelist of Ecstasy and Sin


Author: Vincent Starrett



Release Date: March 7, 2011  [eBook #35515]

Language: English

Character set encoding: ISO-8859-1


***START OF THE PROJECT GUTENBERG EBOOK ARTHUR MACHEN***


E-text prepared by Marc D'Hooghe (http://www.freeliterature.org) from page
images generously made available by Internet Archive
(http://www.archive.org)



Note: Images of the original pages are available through
      Internet A

In [40]:
import joblib
from pathlib import Path
from tqdm import tqdm
import numpy as np
import time
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

MODEL_SAVE = Path("register_clf.joblib")


class RegisterClassifier:

    def __init__(self):
        self._init_vectorizer()
        self._init_model()
        self.trained = False

    # ───────────────────────────────────────────── #
    # CLEAN TEXT (SAFE)
    # ───────────────────────────────────────────── #

    def clean_text(self, text):
        text = text.lower()

        # remove obvious Gutenberg boilerplate
        if "project gutenberg" in text:
            return ""
        if "this ebook is for the use" in text:
            return ""

        # light cleanup only
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    # ───────────────────────────────────────────── #
    # SAFE FORMAL FILTER (DO NOT OVER-FILTER)
    # ───────────────────────────────────────────── #

    def filter_formal(self, texts):
        good = []

        print("\n🔍 Cleaning formal texts...")

        for t in tqdm(texts, desc="Formal Clean"):
            t = self.clean_text(t)

            # keep almost everything meaningful
            if len(t) < 30:
                continue

            if "ebook" in t or "gutenberg" in t:
                continue

            good.append(t)

        return good

    # ───────────────────────────────────────────── #
    # VECTORIZER
    # ───────────────────────────────────────────── #

    def _init_vectorizer(self):
        print("🔧 Initializing vectorizer...")

        self.vec = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=20000,
            stop_words='english'
        )

    # ───────────────────────────────────────────── #
    # MODEL
    # ───────────────────────────────────────────── #

    def _init_model(self):
        print("🧠 Initializing model...")

        self.clf = SGDClassifier(
            loss='log_loss',
            random_state=42
        )

    # ───────────────────────────────────────────── #
    # VECTORIZE
    # ───────────────────────────────────────────── #

    def vectorize(self, X_train, X_test):
        print("\n🔤 TF-IDF vectorizing...")
        t0 = time.time()

        X_train_vec = self.vec.fit_transform(X_train)
        X_test_vec = self.vec.transform(X_test)

        print(f"   Done in {time.time() - t0:.2f}s")
        print(f"   Shape: {X_train_vec.shape}")

        return X_train_vec, X_test_vec

    # ───────────────────────────────────────────── #
    # TRAIN MODEL
    # ───────────────────────────────────────────── #

    def train_model(self, X_train, y_train, X_test, y_test, epochs=5, batch_size=1024):

        print("\n🧠 Training model...")

        y_train = np.array(y_train)
        classes = np.unique(y_train)

        # 🚨 SAFETY CHECK
        if len(classes) < 2:
            raise ValueError("❌ Only one class found. Dataset issue.")

        # class weights
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=classes,
            y=y_train
        )
        class_weight_dict = dict(zip(classes, class_weights))

        for epoch in range(epochs):
            print(f"\n📘 Epoch {epoch+1}/{epochs}")

            indices = np.random.permutation(X_train.shape[0])
            X_train = X_train[indices]
            y_train = y_train[indices]

            losses = []

            for i in tqdm(range(0, X_train.shape[0], batch_size), desc="Training"):

                X_batch = X_train[i:i+batch_size]
                y_batch = y_train[i:i+batch_size]

                weights = np.array([class_weight_dict[y] for y in y_batch])

                self.clf.partial_fit(
                    X_batch,
                    y_batch,
                    classes=classes,
                    sample_weight=weights
                )

                # stable loss
                probs = self.clf.predict_proba(X_batch)
                probs = np.clip(probs, 1e-10, 1.0)

                idxs = [list(self.clf.classes_).index(y) for y in y_batch]
                loss = -np.mean(np.log(probs[np.arange(len(y_batch)), idxs]))
                losses.append(loss)

            # evaluation
            y_pred = self.clf.predict(X_test)
            acc = accuracy_score(y_test, y_pred)

            print(f"   📉 Loss: {np.mean(losses):.4f}")
            print(f"   🎯 Accuracy: {acc:.4f}")

    # ───────────────────────────────────────────── #
    # FULL TRAIN PIPELINE
    # ───────────────────────────────────────────── #

    def train(self, informal_texts, formal_texts):

        print("\n📦 Cleaning dataset...")

        print("\n🧹 Cleaning informal...")
        informal = [self.clean_text(t) for t in tqdm(informal_texts)]

        formal = self.filter_formal(formal_texts)

        print(f"\nAfter cleaning:")
        print(f"Informal: {len(informal)}")
        print(f"Formal: {len(formal)}")

        if len(formal) < 100:
            print("⚠️ Warning: formal dataset small, but continuing anyway...")

        X = informal + formal
        y = ['informal'] * len(informal) + ['formal'] * len(formal)

        # 🚨 FINAL SAFETY CHECK
        if len(set(y)) < 2:
            raise ValueError("❌ Still only one class after processing.")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.15,
            stratify=y,
            random_state=42
        )

        X_train = [t[:300] for t in X_train]
        X_test = [t[:300] for t in X_test]

        print(f"\n🚀 Training on {len(X_train):,} samples...")

        X_train_vec, X_test_vec = self.vectorize(X_train, X_test)

        self.train_model(X_train_vec, y_train, X_test_vec, y_test)

        print("\n📊 Final Evaluation:")
        print(classification_report(y_test, self.clf.predict(X_test_vec)))

        self.trained = True
        return self

    # ───────────────────────────────────────────── #
    # PREDICT (RULE + ML HYBRID)
    # ───────────────────────────────────────────── #

    def predict(self, text):
        if not self.trained:
            raise RuntimeError("Model not trained or loaded.")

        text_low = text.lower()

        # 🔥 rule-based boost
        formal_markers = [
            "therefore", "however", "significant",
            "hypothesis", "analysis", "methodology",
            "results indicate", "conclusion"
        ]

        if any(word in text_low for word in formal_markers):
            return "formal", 0.95

        vec = self.vec.transform([text[:300]])

        proba = self.clf.predict_proba(vec)[0]
        idx = proba.argmax()

        return self.clf.classes_[idx], float(proba[idx])

    # ───────────────────────────────────────────── #
    # SAVE / LOAD
    # ───────────────────────────────────────────── #

    def save(self, path=MODEL_SAVE):
        joblib.dump((self.vec, self.clf), path)
        print(f"💾 Saved to {path}")

    @classmethod
    def load(cls, path=MODEL_SAVE):
        obj = cls()
        obj.vec, obj.clf = joblib.load(path)
        obj.trained = True
        print(f"📂 Loaded from {path}")
        return obj

In [41]:
reg = RegisterClassifier()
reg.train(informal_texts, formal_texts)
reg.save()

🔧 Initializing vectorizer...
🧠 Initializing model...

📦 Cleaning dataset...

🧹 Cleaning informal...


100%|██████████| 15000/15000 [00:00<00:00, 124065.41it/s]



🔍 Cleaning formal texts...


Formal Clean: 100%|██████████| 15000/15000 [00:30<00:00, 498.80it/s]



After cleaning:
Informal: 15000
Formal: 0
⚠️ Warning: formal dataset small, but continuing anyway...


ValueError: ❌ Still only one class after processing.

In [35]:
# LOAD TRAINED MODEL
reg_clf = RegisterClassifier.load("register_clf.joblib")

sanity = [
    ("omg this is literally the best thing ever lol", 'informal'),
    ("just saw my fave band and i cannot stop smiling rn", 'informal'),

    ("I would like to express my sincere appreciation for your assistance.", 'formal'),
    ("The results indicate a significant improvement in performance.", 'formal'),
    ("Please ensure that all requirements are met prior to submission.", 'formal'),

    ("honestly this movie was mid but the vibes were immaculate", 'informal'),
]
print(f"{'Text':<65} {'Expected':<10} {'Predicted':<10} {'Conf':>6}")
print("-" * 95)

correct = 0
for text, expected in sanity:
    label, conf = reg_clf.predict(text)
    tick = '✓' if label == expected else '✗'
    if label == expected:
        correct += 1
    print(f"{tick} {text[:62]:<63} {expected:<10} {label:<10} {conf:>6.3f}")

print(f"\nAccuracy on sanity set: {correct}/{len(sanity)}")

🔧 Initializing vectorizers...
🧠 Initializing model...
📂 Loaded from register_clf.joblib
Text                                                              Expected   Predicted    Conf
-----------------------------------------------------------------------------------------------
✓ omg this is literally the best thing ever lol                   informal   informal    0.979
✓ just saw my fave band and i cannot stop smiling rn              informal   informal    0.992
✗ I would like to express my sincere appreciation for your assis  formal     informal    0.988
✗ The results indicate a significant improvement in performance.  formal     informal    0.965
✗ Please ensure that all requirements are met prior to submissio  formal     informal    0.981
✓ honestly this movie was mid but the vibes were immaculate       informal   informal    0.980

Accuracy on sanity set: 3/6


# Gutenberg is really bad so here's a better set of data to use

In [44]:
from datasets import load_dataset
import pandas as pd
import random

TARGET_PER_SOURCE = 6666
SEED = 42
random.seed(SEED)

# ───────────────────────────────────────────── #
# CLEAN FUNCTION
# ───────────────────────────────────────────── #
def clean_text(t):
    if not t:
        return ""
    t = t.replace("\n", " ").strip()
    if len(t) < 50:
        return ""
    return t


# ───────────────────────────────────────────── #
# GENERIC STREAM SAMPLER
# ───────────────────────────────────────────── #
def stream_sample(dataset, field, k):
    samples = []

    for x in dataset:
        t = clean_text(x[field])
        if t:
            samples.append(t)

        if len(samples) >= k:
            break

    return samples


# ───────────────────────────────────────────── #
# 1. C4
# ───────────────────────────────────────────── #
print("📥 Loading C4...")

c4 = load_dataset("allenai/c4", "en", split="train", streaming=True)
c4_sample = stream_sample(c4, "text", TARGET_PER_SOURCE)

print(f"✅ C4: {len(c4_sample)}")


# ───────────────────────────────────────────── #
# 2. FINEWEB (REPLACES WIKIPEDIA)
# ───────────────────────────────────────────── #
print("📥 Loading FineWeb...")

fineweb = load_dataset("HuggingFaceFW/fineweb", split="train", streaming=True)
fineweb_sample = stream_sample(fineweb, "text", TARGET_PER_SOURCE)

print(f"✅ FineWeb: {len(fineweb_sample)}")


# ───────────────────────────────────────────── #
# 3. CNN / DAILYMAIL
# ───────────────────────────────────────────── #
print("📥 Loading CNN/DailyMail...")

news = load_dataset("cnn_dailymail", "3.0.0", split="train", streaming=True)
news_sample = stream_sample(news, "article", TARGET_PER_SOURCE)

print(f"✅ News: {len(news_sample)}")


# ───────────────────────────────────────────── #
# COMBINE
# ───────────────────────────────────────────── #
print("📦 Combining datasets...")

texts = c4_sample + fineweb_sample + news_sample

sources = (
    ["c4"] * TARGET_PER_SOURCE +
    ["fineweb"] * TARGET_PER_SOURCE +
    ["news"] * TARGET_PER_SOURCE
)

labels = ["formal"] * len(texts)

df = pd.DataFrame({
    "text": texts,
    "label": labels,
    "source": sources
})


# ───────────────────────────────────────────── #
# SHUFFLE
# ───────────────────────────────────────────── #
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)


# ───────────────────────────────────────────── #
# SAVE AS PARQUET
# ───────────────────────────────────────────── #
output_path = "formal_20k.parquet"
df.to_parquet(output_path, index=False)

print(f"\n✅ Saved {len(df)} samples to {output_path}")
print(df.head())

📥 Loading C4...
✅ C4: 6666
📥 Loading FineWeb...
✅ FineWeb: 6666
📥 Loading CNN/DailyMail...
✅ News: 6666
📦 Combining datasets...

✅ Saved 19998 samples to formal_20k.parquet
                                                text   label   source
0  (CNN) -- Emergency crews called off a search i...  formal     news
1  The ‘start Trainer Of Kg’ : Redefines What Can...  formal       c4
2  Understanding how landscape factors, including...  formal       c4
3  If you want to really enjoy your fly fishing e...  formal  fineweb
4  For the seventh year, the Academy of Country M...  formal  fineweb


In [60]:
# ---------------------------
# FULL WORKING CLASSIFIER
# ---------------------------

import joblib
from pathlib import Path
import re
import pandas as pd
import numpy as np
import chardet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import FeatureUnion

MODEL_SAVE = Path("register_clf.joblib")


class RegisterClassifier:

    def __init__(self):
        self._init_vectorizer()
        self._init_model()
        self.trained = False

    def clean_text(self, text):
        if not text:
            return ""
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _init_vectorizer(self):
        word_vec = TfidfVectorizer(ngram_range=(1,2), max_features=15000, stop_words='english')
        char_vec = TfidfVectorizer(analyzer='char', ngram_range=(3,5), max_features=10000)
        self.vec = FeatureUnion([("word", word_vec), ("char", char_vec)])

    def _init_model(self):
        self.clf = SGDClassifier(loss='log_loss', random_state=42)

    def vectorize(self, X_train, X_test):
        print("Vectorizing data...")
        X_train_vec = self.vec.fit_transform(X_train)
        X_test_vec = self.vec.transform(X_test)
        print(f"Done. Shape: {X_train_vec.shape}")
        return X_train_vec, X_test_vec

    def train_model(self, X_train, y_train, X_test, y_test, epochs=5, batch_size=1024):
        y_train = np.array(y_train)
        classes = np.unique(y_train)
        class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
        class_weight_dict = dict(zip(classes, class_weights))

        n_batches = int(np.ceil(X_train.shape[0] / batch_size))

        for epoch in range(epochs):
            indices = np.random.permutation(X_train.shape[0])
            X_train_shuff = X_train[indices]
            y_train_shuff = y_train[indices]
            losses = []

            print(f"\nEpoch {epoch+1}/{epochs}")

            for i in range(n_batches):
                start = i*batch_size
                end = min(start + batch_size, X_train.shape[0])
                X_batch = X_train_shuff[start:end]
                y_batch = y_train_shuff[start:end]
                weights = np.array([class_weight_dict[y] for y in y_batch])

                self.clf.partial_fit(X_batch, y_batch, classes=classes, sample_weight=weights)

                probs = self.clf.predict_proba(X_batch)
                probs = np.clip(probs, 1e-10, 1.0)
                idxs = [list(self.clf.classes_).index(y) for y in y_batch]
                batch_loss = -np.mean(np.log(probs[np.arange(len(y_batch)), idxs]))
                losses.append(batch_loss)

                # **GUARANTEED PROGRESS PRINT**
                print(f"  Batch {i+1}/{n_batches} — Loss: {np.mean(losses):.4f}")

            acc = accuracy_score(y_test, self.clf.predict(X_test))
            print(f"🎯 Epoch {epoch+1} Accuracy: {acc:.4f}")

    def train(self, informal_texts, formal_texts):
        print("\nCleaning informal texts...")
        informal_clean = []
        for i, t in enumerate(informal_texts, 1):
            clean = self.clean_text(t)
            if clean:
                informal_clean.append(clean)
            if i % 500 == 0 or i == len(informal_texts):
                print(f"  {i}/{len(informal_texts)} informal texts processed")

        print("\nCleaning formal texts...")
        formal_clean = []
        for i, t in enumerate(formal_texts, 1):
            clean = self.clean_text(t)
            if clean:
                formal_clean.append(clean)
            if i % 500 == 0 or i == len(formal_texts):
                print(f"  {i}/{len(formal_texts)} formal texts processed")

        print(f"\nTotal: Informal={len(informal_clean)}, Formal={len(formal_clean)}")

        X = informal_clean + formal_clean
        y = ['informal']*len(informal_clean) + ['formal']*len(formal_clean)

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
        X_train = [t[:300] for t in X_train]
        X_test = [t[:300] for t in X_test]

        X_train_vec, X_test_vec = self.vectorize(X_train, X_test)
        self.train_model(X_train_vec, y_train, X_test_vec, y_test)

        print("\nFinal Evaluation:")
        print(classification_report(y_test, self.clf.predict(X_test_vec)))
        self.trained = True
        return self

    def predict(self, text):
        if not self.trained:
            raise RuntimeError("Model not trained.")
        vec = self.vec.transform([text[:300]])
        proba = self.clf.predict_proba(vec)[0]
        idx = proba.argmax()
        return self.clf.classes_[idx], float(proba[idx])

    def save(self, path=MODEL_SAVE):
        joblib.dump((self.vec, self.clf), path)
        print(f"Saved model to {path}")

    @classmethod
    def load(cls, path=MODEL_SAVE):
        obj = cls()
        obj.vec, obj.clf = joblib.load(path)
        obj.trained = True
        print(f"Loaded model from {path}")
        return obj


# ---------------------------
# HELPER FUNCTION
# ---------------------------
def read_csv_autodetect(path):
    with open(path, "rb") as f:
        result = chardet.detect(f.read())
    print(f"Detected encoding: {result['encoding']}")
    return pd.read_csv(path, encoding=result['encoding'], errors='replace')


# ---------------------------
# LOAD DATA
# ---------------------------
df_formal = pd.read_parquet("formal_20k.parquet")
formal_texts = df_formal["text"].tolist()

df_informal = read_csv_autodetect("/Users/yatharthnehva/NLPproject/characterlevelattacks/emojibased/twittersenti/twittersenti140.csv")
informal_texts = df_informal["text"].tolist()


# ---------------------------
# TRAIN
# ---------------------------
reg = RegisterClassifier()
reg.train(informal_texts, formal_texts)
reg.save()

Detected encoding: Windows-1252


TypeError: read_csv() got an unexpected keyword argument 'errors'

In [33]:
print("Informal:", len(informal_texts))
print("Formal:", len(formal_texts))

Informal: 15000
Formal: 15000


In [34]:
for t in formal_texts[:5]:
    print("\n", t[:200])


 The Project Gutenberg eBook, Arthur Machen, by Vincent Starrett


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or


 The Project Gutenberg eBook, Famous Authors (Men), by E. F. (Edward
Francis) Harkins


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy

 ﻿Project Gutenberg's The Lenâpé and their Legends, by Daniel G. Brinton

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it a

 The Project Gutenberg eBook, Fenton's Quest, by M. E. Braddon


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re

 ﻿The Project Gutenberg EBook of Essays Literary, Critical and Historical, by 
Thomas O'Hagan

This eBook is for the use of anyone anywhere in the United States and most
other parts of th